In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import pandas as pd
from pathlib import Path

In [26]:
base_dir = '/Users/amonast/Desktop/miniscope/data'

In [27]:
# ── Mapping: Observation N → r_day + full animal name (C3 cohort) ─────────────
obs_map = pd.DataFrame({
    "obs_num": [ 1,     2,     3,     4,     5,     6,     7,     8,     9,    10,    11,    12],
    "r_day":   [ 1,     1,     1,     1,     1,     1,     2,     2,     2,     2,     2,     2],
    "animal":  ["C3F2","C3M2","C3F3","C3M3","C3F4","C3M4","C3F2","C3M2","C3F3","C3M3","C3F4","C3M4"],
})

# ── Load Event Log (C3) ───────────────────────────────────────────────────────
df1 = pd.read_csv(os.path.join(base_dir,"C3/dbnst_endo_pan_c5 - Data Profile - Event Logs.csv"))

df1["obs_num"] = df1["Observation"].str.extract(r"(\d+)$").astype(int)
df1 = df1.merge(obs_map, on="obs_num", how="left")
df1["session"] = "R" + df1["r_day"].astype(str)
df1 = df1.drop(columns=["obs_num", "r_day"])

# ── Load C0/C1/C2 xlsx files ──────────────────────────────────────────────────
# C0 currently holds one file: "dbnst_gcamp6f_vmh_retro_f1_day3 - C0F2C - R2.xlsx"
# whose stem splits (on " - ") into animal=C0F2C, session=R2 — same naming
# convention as the C1/C2 files, so it's picked up by the existing loop.

dfs = []
for subdir in ["C1", "C2", "C0"]:
    for fpath in (Path(base_dir) / subdir).glob("*.[cx][sl][vs]*"):
        if fpath.name.startswith("~$"):   # skip Excel lock/temp files
            continue
        parts = fpath.stem.split(" - ")
        df = pd.read_excel(fpath) if fpath.suffix == ".xlsx" else pd.read_csv(fpath)
        df["animal"] = parts[-2]   # e.g. C1F2, C2M3, C0F2C
        df["session"] = parts[-1]  # e.g. R1, R2
        dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

# ── Stack all cohorts ─────────────────────────────────────────────────────────
all_data = pd.concat([combined, df1], ignore_index=True)

In [28]:
df = all_data[['animal','session','Time_Relative_hmsf','Behavior','Event_Type','Duration_sf','Observation']]

In [29]:
df['time']=df['Time_Relative_hmsf'].apply(pd.to_timedelta)
df

/var/folders/4k/5kmnnrq94j136038_jy_4twc0000gq/T/ipykernel_6224/1422584006.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time']=df['Time_Relative_hmsf'].apply(pd.to_timedelta)


,animal,session,Time_Relative_hmsf,Behavior,Event_Type,Duration_sf,Observation,time
0,C1M1,R1,0 days 00:02:16.393000,Thigmotaxis (without freezing),State start,2.73454,Observation 1,0 days 00:02:16.393000
1,C1M1,R1,0 days 00:02:19.128000,Thigmotaxis (without freezing),State stop,0.00000,Observation 1,0 days 00:02:19.128000
2,C1M1,R1,0 days 00:02:19.295000,Face Sniffing,State start,3.13471,Observation 1,0 days 00:02:19.295000
3,C1M1,R1,0 days 00:02:22.429000,Face Sniffing,State stop,0.00000,Observation 1,0 days 00:02:22.429000
4,C1M1,R1,0 days 00:02:22.563000,Body Sniffing,State start,0.33348,Observation 1,0 days 00:02:22.563000
...,...,...,...,...,...,...,...,...
8881,C3F3,R2,00:12:16.457,Face Sniffing,State stop,0.00000,Observation 9,0 days 00:12:16.457000
8882,C3F3,R2,00:12:17.191,Digging,State start,2.15311,Observation 9,0 days 00:12:17.191000
8883,C3F3,R2,00:12:18.592,NaN,State point,0.00000,Observation 9,0 days 00:12:18.592000
8884,C3F3,R2,00:12:19.311,NaN,State point,0.00000,Observation 9,0 days 00:12:19.311000


In [30]:
df.animal.unique()

array(['C1M1', 'C1F2', 'C1M2', 'C2F1', 'C2M4', 'C0F2C', 'C3F2', 'C3M3',
       'C3F4', 'C3M4', 'C3M2', 'C3F3'], dtype=object)

In [31]:
def get_intervals(group):
    starts = group[group["Event_Type"] == "State start"].reset_index(drop=True)
    stops  = group[group["Event_Type"] == "State stop"].reset_index(drop=True)

    return pd.DataFrame({
        "Behavior": starts["Behavior"],
        "Interval": pd.IntervalIndex.from_arrays(
                        starts["time"].dt.total_seconds(),
                        stops["time"].dt.total_seconds(),
                        closed="both"),}).assign(
                            start=lambda df: df["Interval"].apply(lambda x: x.left),
                            end=lambda df:   df["Interval"].apply(lambda x: x.right),
                            ).sort_values("start").reset_index(drop=True)

interval_dict = {obs: get_intervals(grp)
    for obs, grp in df.groupby("Observation")}

observations = df["Observation"].unique()
interval_dict = {obs: get_intervals(grp)
    for obs, grp in df.groupby("Observation")}


In [32]:
def find_overlapping_intervals(intervals):
    intervals["start"] = intervals["Interval"].apply(lambda x: x.left)
    intervals["end"]   = intervals["Interval"].apply(lambda x: x.right)

    # Sort by start time (important)
    intervals = intervals.sort_values("start").reset_index(drop=True)

    # Find overlaps
    overlap_mask = intervals["start"] < intervals["end"].shift(1)
    overlapping = intervals.loc[overlap_mask | overlap_mask.shift(-1).fillna(False)]

    return overlapping

def merge_and_clip_intervals(overlapping):
    intervals = overlapping.copy().sort_values("start").reset_index(drop=True)
    result = []
    i = 0
    while i < len(intervals):
        current = intervals.iloc[i].copy()
        j = i + 1
        while j < len(intervals):
            nxt = intervals.iloc[j]
            if nxt["start"] >= current["end"]:
                break
            if nxt["Behavior"] == current["Behavior"]:
                current["end"] = max(current["end"], nxt["end"])
                intervals.at[j, "start"] = current["end"]
                j += 1
            else:
                current["end"] = nxt["start"]
                break
        result.append(current)
        i = j if (j > i + 1) else i + 1
    out = pd.DataFrame(result).reset_index(drop=True)
    out["Interval"] = pd.arrays.IntervalArray.from_arrays(
        out["start"], out["end"], closed="both")
    return out[["Behavior", "Interval", "start", "end"]]



In [33]:
overlaps_all = {obs: find_overlapping_intervals(grp)for obs, grp in interval_dict.items()}
overlaps_all

{'Observation 1':                            Behavior            Interval    start      end
 0    Thigmotaxis (without freezing)  [136.393, 139.128]  136.393  139.128
 1                            Escape  [138.494, 139.228]  138.494  139.228
 2                     Face Sniffing  [139.295, 142.429]  139.295  142.429
 3                     Body Sniffing  [139.828, 140.428]  139.828  140.428
 8                     Face Sniffing  [144.063, 145.331]  144.063  145.331
 ..                              ...                 ...      ...      ...
 707                         Digging    [757.733, 758.1]  757.733  758.100
 709                         Digging  [759.601, 768.038]  759.601  768.038
 710                   Body Sniffing  [761.368, 761.635]  761.368  761.635
 717                         Digging   [766.57, 780.297]  766.570  780.297
 718                         Digging  [768.838, 770.972]  768.838  770.972
 
 [557 rows x 4 columns],
 'Observation 10':                             Behavior 

# Plot Behavior Rasters

In [34]:
behaviors = df["Behavior"].unique()
behaviors

array(['Thigmotaxis (without freezing)', 'Face Sniffing', 'Body Sniffing',
       'Tail Sniffing', 'Anogenital Sniffing', 'Digging', nan,
       'Self-Grooming', 'Escape', 'Chasing', 'Mounting',
       'Other Behavior of Interest', 'Attack', 'Grooming',
       'Freezing (social)', 'Freezing (non-social)',
       'White Mouse Aggression', 'Cord Grabbing', 'Aggression (Misc.)',
       'Social Avoidance (Misc.)', 'Aggressive Grooming/Mounting',
       'Aggressive Grooming/Barbing', 'Tail-Rattling',
       'White Mouse Behavior of Interest', 'Non-Prosocial (Misc.)'],
      dtype=object)

In [35]:
# ── Sub-behavior color map (colors corrected to match categories) ─────────────
color_map_subtypes = {
    # Aggressive — reds/oranges
    'Attack':                           '#C62828',
    'Aggressive Grooming/Barbing':      '#E53935',
    'Aggressive Grooming/Mounting':     '#E53935',
    'Tail-Rattling':                    '#FF5722',
    'White Mouse Aggression':           '#BF360C',
    'Mounting':                         '#EF6C00',  # fixed: was teal
    'Aggression (Misc.)':               '#FF7043',

    # Pro-Social — greens/teals
    'Tail Sniffing':                    '#2E7D32',
    'Body Sniffing':                    '#43A047',
    'Anogenital Sniffing':              '#66BB6A',
    'Face Sniffing':                    '#26A69A',
    'Grooming':                         '#00897B',

    # Social Avoidance — purples
    'Freezing (social)':                '#6A1B9A',  
    'Freezing (non-social)':            '#8E24AA',  
    'Social Avoidance (Misc.)':         '#9C27B0',  
    'Escape':                           "#C15EEC", 
    'Digging':                          "#882A99",  
    'Thigmotaxis (without freezing)':   '#7E57C2',

    # Other — grays
    'White Mouse Behavior of Interest': '#444042',
    'Non-Prosocial (Misc.)':            '#90A4AE',
    'Self-Grooming':                    '#757575',  # fixed: was purple
    'Cord Grabbing':                    '#BDBDBD',  # fixed: was purple
}

# ── 4-category color map ──────────────────────────────────────────────────────
color_map_categories = {
    'Aggressive':        "#C54A4A",
    'Pro-Social':        "#57A35B",
    'Social Avoidance':  '#7B1FA2',
    'Other':             '#78909C',
}

# ── Classification function ───────────────────────────────────────────────────
_aggressive = {
    'Attack', 'Aggressive Grooming/Barbing', 'Tail-Rattling',
    'White Mouse Aggression', 'Mounting', 'Aggression (Misc.)',
    'Aggressive Grooming/Mounting',
}

_prosocial = {
    'Tail Sniffing', 'Body Sniffing', 'Anogenital Sniffing',
    'Face Sniffing', 'Grooming',
}
_avoidance = {
    'Freezing (social)', 'Freezing (non-social)', 'Social Avoidance (Misc.)',
    'Escape', 'Digging', 'Thigmotaxis (without freezing)',
}

def classify_behavior(behavior: str) -> str:
    if behavior in _aggressive:
        return 'Aggressive'
    if behavior in _prosocial:
        return 'Pro-Social'
    if behavior in _avoidance:
        return 'Social Avoidance'
    return 'Other'

In [36]:
df['behavior_type']=df['Behavior'].apply(classify_behavior)

/var/folders/4k/5kmnnrq94j136038_jy_4twc0000gq/T/ipykernel_6224/316246701.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['behavior_type']=df['Behavior'].apply(classify_behavior)


In [37]:
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf
import pandas as pd
from pathlib import Path

def build_intervals(session_df: pd.DataFrame, 
                    group_col: str = "Behavior") -> pd.DataFrame:
    intervals = []
    for value, grp in session_df.groupby(group_col):
        starts = grp[grp["Event_Type"] == "State start"]["time"].dt.total_seconds().to_numpy()
        stops  = grp[grp["Event_Type"] == "State stop" ]["time"].dt.total_seconds().to_numpy()
        for s, e in zip(starts, stops):
            intervals.append({group_col: value, "start": s, "end": e})
    return pd.DataFrame(intervals) if intervals else pd.DataFrame(columns=[group_col, "start", "end"])


def plot_animal_timeline(animal: str,
                         df: pd.DataFrame,
                         color_map: dict,
                         output_dir: str | Path,
                         group_col: str = "Behavior",
                         filename_suffix: str = "") -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    animal_df = df[df["animal"] == animal]

    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

    for ax, session in zip(axes, ["R1", "R2"]):
        session_df = animal_df[animal_df["session"] == session]
        intervals  = build_intervals(session_df, group_col=group_col)

        seen_labels = set()
        for _, row in intervals.iterrows():
            label = row[group_col] if row[group_col] not in seen_labels else ""
            seen_labels.add(row[group_col])

            ax.barh(
                y=0,
                width=row["end"] - row["start"],
                left=row["start"],
                height=0.8,
                alpha=0.6,
                color=color_map.get(row[group_col], "#aaaaaa"),
                label=label,
            )

        ax.set_ylim([-0.5, 0.5])
        ax.set_xlim([0, 700])
        ax.vlines(120, -0.5, 0.5, color="r", linestyles="dashed", linewidth=1)
        ax.set_ylabel(session, fontsize=10, rotation=0, labelpad=40, va="center")
        ax.set_yticks([])
        ax.spines[["top", "right", "left"]].set_visible(False)

        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(
                handles, labels,
                title=group_col,
                frameon=False,
                loc="center left",
                bbox_to_anchor=(1.01, 0.5),
                fontsize=8,
            )

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Behavior Timeline — {animal}", y=1.01)
    plt.tight_layout()

    fname = f"{animal}{('_' + filename_suffix) if filename_suffix else ''}.pdf"
    fig.savefig(output_dir / fname, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {output_dir / fname}")

In [38]:
# ── Run for every animal ──────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/amonast/Desktop/miniscope/Behavior Plots"
for animal in df["animal"].unique():
    plot_animal_timeline(animal, df, color_map_subtypes, OUTPUT_DIR,group_col='Behavior')
    plot_animal_timeline(animal, df, color_map_categories, OUTPUT_DIR,group_col='behavior_type',filename_suffix='categories')

Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C1M1.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C1M1_categories.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C1F2.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C1F2_categories.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C1M2.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C1M2_categories.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C2F1.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C2F1_categories.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C2M4.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C2M4_categories.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C0F2C.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C0F2C_categories.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C3F2.pdf
Saved: /Users/amonast/Desktop/miniscope/Behavior Plots/C3F2_categories.pdf
Saved: /Users/amonast/Des

In [39]:
df.columns

Index(['animal', 'session', 'Time_Relative_hmsf', 'Behavior', 'Event_Type',
       'Duration_sf', 'Observation', 'time', 'behavior_type'],
      dtype='object')

In [40]:
df2 = df.drop(['Duration_sf','Observation','Time_Relative_hmsf'],axis=1)
df2

,animal,session,Behavior,Event_Type,time,behavior_type
0,C1M1,R1,Thigmotaxis (without freezing),State start,0 days 00:02:16.393000,Social Avoidance
1,C1M1,R1,Thigmotaxis (without freezing),State stop,0 days 00:02:19.128000,Social Avoidance
2,C1M1,R1,Face Sniffing,State start,0 days 00:02:19.295000,Pro-Social
3,C1M1,R1,Face Sniffing,State stop,0 days 00:02:22.429000,Pro-Social
4,C1M1,R1,Body Sniffing,State start,0 days 00:02:22.563000,Pro-Social
...,...,...,...,...,...,...
8881,C3F3,R2,Face Sniffing,State stop,0 days 00:12:16.457000,Pro-Social
8882,C3F3,R2,Digging,State start,0 days 00:12:17.191000,Social Avoidance
8883,C3F3,R2,NaN,State point,0 days 00:12:18.592000,Other
8884,C3F3,R2,NaN,State point,0 days 00:12:19.311000,Other


In [41]:
df2.loc[df2.animal=='C0F2C'].behavior_type.unique()

array(['Pro-Social', 'Social Avoidance', 'Other', 'Aggressive'],
      dtype=object)

In [42]:
df2.to_csv("/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv")